# Proxy-Shuffle Negative Control

Runs the full 20-seed real-data proxy-shuffle negative control and saves summary tables only.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "experiments").exists() and (p / "config").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.stratified_kstar import build_economics_stratifiers, build_energy_stratifiers
from experiments.run_complete_20seed_suite import SEEDS as COMPLETE_SEEDS
from experiments.run_proxy_shuffle_negative_control import _run_economics, _run_energy
from experiments.run_proxy_shuffle_stratified_kstar import ECON_RAW, ENERGY_RAW, ORIGINAL_ECON_ROOT, ORIGINAL_ENERGY_ROOT, _run_domain

SEEDS = list(range(20))
OUTPUT_ROOT = repo_root / "outputs" / "negative_controls" / "proxy_shuffle_20seed_20260511"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
FORCE = False
DRY_RUN = False
SMOKE = False
PROXY_PERM_SEED_OFFSET = 10000
N_PERM = 2000
BOOTSTRAP_SAMPLES = 10000
RUN_PROXY_SHUFFLE = True
RUN_STRATIFIED_AUDIT = True

invalid = [seed for seed in SEEDS if seed not in COMPLETE_SEEDS]
if invalid:
    raise ValueError(f"Seeds outside the supported complete-suite range: {invalid}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

def clean_incomplete_proxy_runs() -> None:
    specs = [
        (OUTPUT_ROOT / "economics" / "cmdl", "economics_proxy_shuffle_seed"),
        (OUTPUT_ROOT / "energy" / "cmdl", "energy_proxy_shuffle_seed"),
    ]
    for root, prefix in specs:
        if not root.exists():
            continue
        for seed in SEEDS:
            run_dir = root / f"{prefix}{seed}"
            if run_dir.exists() and not (run_dir / "summary.json").exists():
                print(f"[clean] incomplete artifact: {run_dir}")
                shutil.rmtree(run_dir)

clean_incomplete_proxy_runs()
display(pd.Series({"seeds": SEEDS, "output_root": OUTPUT_ROOT, "force": FORCE, "smoke": SMOKE, "n_perm": N_PERM}).to_frame("value"))

In [ ]:
if RUN_PROXY_SHUFFLE:
    _run_economics(OUTPUT_ROOT, SEEDS, force=FORCE, dry_run=DRY_RUN, seed_offset=PROXY_PERM_SEED_OFFSET, smoke=SMOKE)
    _run_energy(OUTPUT_ROOT, SEEDS, force=FORCE, dry_run=DRY_RUN, seed_offset=PROXY_PERM_SEED_OFFSET, smoke=SMOKE)
else:
    print("RUN_PROXY_SHUFFLE = False; using existing proxy-shuffle artifacts if present.")

In [ ]:
summaries = []
tables = {}
if RUN_STRATIFIED_AUDIT:
    econ_per_seed, econ_aggregated, econ_summary = _run_domain(
        "economics",
        ORIGINAL_ECON_ROOT,
        OUTPUT_ROOT,
        ECON_RAW,
        build_economics_stratifiers,
        COMPARISON_DIR,
        n_perm=N_PERM,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    energy_per_seed, energy_aggregated, energy_summary = _run_domain(
        "energy",
        ORIGINAL_ENERGY_ROOT,
        OUTPUT_ROOT,
        ENERGY_RAW,
        build_energy_stratifiers,
        COMPARISON_DIR,
        n_perm=N_PERM,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    combined_summary = pd.concat([econ_summary, energy_summary], ignore_index=True, sort=False)
    combined_summary.to_csv(COMPARISON_DIR / "proxy_shuffle_summary.csv", index=False)
    tables = {
        "economics_proxy_shuffle_stratified_kstar_per_seed": econ_per_seed,
        "economics_proxy_shuffle_stratified_kstar_aggregated": econ_aggregated,
        "economics_proxy_shuffle_summary": econ_summary,
        "energy_proxy_shuffle_stratified_kstar_per_seed": energy_per_seed,
        "energy_proxy_shuffle_stratified_kstar_aggregated": energy_aggregated,
        "energy_proxy_shuffle_summary": energy_summary,
        "proxy_shuffle_summary": combined_summary,
    }
else:
    print("RUN_STRATIFIED_AUDIT = False; using existing comparison tables if present.")
    for path in COMPARISON_DIR.glob("*.csv"):
        tables[path.stem] = pd.read_csv(path)

pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows").to_frame()

In [ ]:
for name, frame in tables.items():
    print(f"\n=== {name} ({len(frame)} rows) ===")
    display(frame.head(20))